# 🎯 Background Removal AI — Complete Training Notebook

**Train a custom MobileNetV3 U-Net with CBAM attention for background removal.**

Architecture: MobileNetV3-Small Encoder + CBAM Attention Decoder + Deep Supervision

| Component | Details |
|---|---|
| Model | MobileNetV3 U-Net (~3.8M params) |
| Loss | BCE + Dice + Focal + IoU (combined) |
| Optimizer | AdamW (lr=1e-4, wd=1e-4) |
| Schedule | Linear Warmup + Cosine Annealing |
| Training | AMP + Gradient Accumulation |
| Target | IoU > 0.85 after 70 epochs |

**Expected Training Time on Colab T4:**
- ~10,000 samples: ~15 min/epoch → ~17.5 hours total (70 epochs)
- Checkpointed every epoch → resume after disconnect

---

## Section A — Setup & Environment

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Project directory on Drive
PROJECT_DIR = '/content/drive/MyDrive/bg_removal_ai'
!mkdir -p {PROJECT_DIR}
print(f'Project directory: {PROJECT_DIR}')

In [ ]:
# Verify GPU availability
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('⚠ No GPU detected! Go to Runtime → Change runtime type → GPU')

In [ ]:
# Install dependencies
!pip install -q albumentations>=1.3.0 onnx>=1.14.0 onnxruntime>=1.15.0

# Verify
import albumentations
import numpy as np
import cv2
from PIL import Image
print(f'Albumentations: {albumentations.__version__}')
print(f'NumPy: {np.__version__}')
print(f'OpenCV: {cv2.__version__}')

In [ ]:
# Set all random seeds for reproducibility
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'All seeds set to {SEED}')

In [ ]:
# Clone or upload project code
# Option 1: Upload zip from local machine
# from google.colab import files
# uploaded = files.upload()  # Upload bg_removal_ai.zip
# !unzip -q bg_removal_ai.zip -d /content/

# Option 2: Clone from GitHub (if you've pushed the code)
# !git clone https://github.com/YOUR_USERNAME/bg_removal_ai.git /content/bg_removal_ai

# Option 3: Copy from Drive (if previously uploaded)
!cp -r {PROJECT_DIR}/code /content/bg_removal_ai 2>/dev/null || echo 'Upload code first'

import sys
sys.path.insert(0, '/content/bg_removal_ai')
print('Project code loaded')

## Section B — Dataset Preparation

In [ ]:
# Download DUTS-TR dataset (10,553 images, ~380MB total)
# This is the primary training dataset - salient object detection

DATASET_DIR = '/content/dataset'
RAW_DIR = '/content/raw_data'

!mkdir -p {RAW_DIR}/DUTS-TR

# Download DUTS-TR images and masks
# Note: If direct download fails, manually download from:
# http://saliencydetection.net/duts/

import os
if not os.path.exists(f'{RAW_DIR}/DUTS-TR/DUTS-TR-Image'):
    print('Downloading DUTS-TR images...')
    !wget -q --show-progress -O /content/DUTS-TR-Image.zip \
        'http://saliencydetection.net/duts/download/DUTS-TR-Image.zip'
    !unzip -q /content/DUTS-TR-Image.zip -d {RAW_DIR}/DUTS-TR/
    !rm /content/DUTS-TR-Image.zip
    print('✓ Images downloaded')

if not os.path.exists(f'{RAW_DIR}/DUTS-TR/DUTS-TR-Mask'):
    print('Downloading DUTS-TR masks...')
    !wget -q --show-progress -O /content/DUTS-TR-Mask.zip \
        'http://saliencydetection.net/duts/download/DUTS-TR-Mask.zip'
    !unzip -q /content/DUTS-TR-Mask.zip -d {RAW_DIR}/DUTS-TR/
    !rm /content/DUTS-TR-Mask.zip
    print('✓ Masks downloaded')

print(f'\nRaw data directory contents:')
!ls -la {RAW_DIR}/DUTS-TR/

In [ ]:
# Prepare dataset: merge + split (80/10/10)
from data.prepare_data import prepare_all

prepare_all(
    raw_dir=RAW_DIR,
    output_dir=DATASET_DIR,
    seed=42,
)

# Verify split sizes
for split in ['train', 'val', 'test']:
    img_dir = f'{DATASET_DIR}/{split}/images'
    if os.path.isdir(img_dir):
        n = len(os.listdir(img_dir))
        print(f'  {split}: {n} samples')

In [ ]:
# Visualize sample data
import matplotlib.pyplot as plt
import cv2
import numpy as np

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

img_dir = f'{DATASET_DIR}/train/images'
mask_dir = f'{DATASET_DIR}/train/masks'
samples = sorted(os.listdir(img_dir))[:3]

for i, fname in enumerate(samples):
    stem = os.path.splitext(fname)[0]
    
    img = cv2.imread(f'{img_dir}/{fname}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Find matching mask
    mask = None
    for ext in ['.png', '.jpg']:
        mpath = f'{mask_dir}/{stem}{ext}'
        if os.path.exists(mpath):
            mask = cv2.imread(mpath, cv2.IMREAD_GRAYSCALE)
            break
    
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'Image: {fname}')
    axes[i, 0].axis('off')
    
    if mask is not None:
        axes[i, 1].imshow(mask, cmap='gray')
        axes[i, 1].set_title('Mask')
        axes[i, 1].axis('off')
        
        # Overlay
        overlay = img.copy()
        overlay[mask < 128] = overlay[mask < 128] // 2
        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title('Overlay')
        axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## Section C — Model Architecture

In [ ]:
# Build and validate model
from model.unet import build_model

model = build_model(320)
params = model.get_param_count()

print('=' * 60)
print('Model Architecture Summary')
print('=' * 60)
for k, v in params.items():
    if isinstance(v, int):
        print(f'  {k}: {v:,}')
    else:
        print(f'  {k}: {v:.1f}')

# Test forward pass
x = torch.randn(2, 3, 320, 320)
model.train()
outputs = model(x)
print(f'\nTraining outputs: {len(outputs)} tensors')
for i, o in enumerate(outputs):
    print(f'  S{i+1}: {o.shape}')

model.eval()
with torch.no_grad():
    out = model(x)
print(f'\nInference output: {out.shape}')
print('\n✓ Model validated!')

## Section D — Training Configuration

In [ ]:
# Training hyperparameters
CONFIG = {
    # Data
    'input_size': 320,
    'batch_size': 4,           # Fits in T4 15GB VRAM with AMP
    'accumulation_steps': 4,   # Effective batch = 16
    'num_workers': 2,
    
    # Optimizer (AdamW)
    'lr': 1e-4,               # α = learning rate
    'weight_decay': 1e-4,     # λ = decoupled weight decay
    'beta1': 0.9,             # β₁ = momentum decay
    'beta2': 0.999,           # β₂ = RMS decay
    
    # Schedule
    'warmup_epochs': 5,       # Linear warmup
    'total_epochs': 70,       # Full training
    'T_0': 10,                # Initial cosine period
    'T_mult': 2,              # Period multiplier
    'min_lr': 1e-6,           # Minimum LR
    
    # Loss weights
    'bce_weight': 0.3,        # λ₁
    'dice_weight': 0.4,       # λ₂ (most important)
    'focal_weight': 0.2,      # λ₃
    'iou_weight': 0.1,        # λ₄
    
    # Training control
    'early_stopping_patience': 15,
    'gradient_clip_norm': 1.0,
    
    # Paths
    'data_dir': DATASET_DIR,
    'checkpoint_dir': f'{PROJECT_DIR}/checkpoints',
}

print('Training Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## Section E — Training Loop

In [ ]:
import time
import json
from torch.cuda.amp import autocast, GradScaler
from tqdm.notebook import tqdm

from model.unet import build_model
from model.loss import CombinedLoss, DeepSupervisionLoss
from data.dataset import SegmentationDataset, create_data_loaders
from data.augmentations import get_train_transforms, get_val_transforms
from train import (
    set_seed, compute_iou, compute_dice,
    WarmupCosineScheduler, CheckpointManager,
)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

# ── Model ──
model = build_model(CONFIG['input_size']).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

# ── Data ──
train_loader, val_loader = create_data_loaders(
    f"{CONFIG['data_dir']}/train/images",
    f"{CONFIG['data_dir']}/train/masks",
    f"{CONFIG['data_dir']}/val/images",
    f"{CONFIG['data_dir']}/val/masks",
    get_train_transforms(CONFIG['input_size']),
    get_val_transforms(CONFIG['input_size']),
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
)
print(f'Train: {len(train_loader.dataset)} samples')
print(f'Val:   {len(val_loader.dataset)} samples')

# ── Loss ──
base_criterion = CombinedLoss(
    bce_weight=CONFIG['bce_weight'],
    dice_weight=CONFIG['dice_weight'],
    focal_weight=CONFIG['focal_weight'],
    iou_weight=CONFIG['iou_weight'],
)
criterion = DeepSupervisionLoss(criterion=base_criterion)

# ── Optimizer ──
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['lr'],
    betas=(CONFIG['beta1'], CONFIG['beta2']),
    weight_decay=CONFIG['weight_decay'],
)

# ── Scheduler ──
scheduler = WarmupCosineScheduler(
    optimizer, CONFIG['warmup_epochs'],
    CONFIG['T_0'], CONFIG['T_mult'],
    CONFIG['min_lr'], CONFIG['total_epochs'],
)

# ── AMP ──
scaler = GradScaler(enabled=(device.type == 'cuda'))

# ── Checkpoint ──
ckpt_manager = CheckpointManager(CONFIG['checkpoint_dir'])

print('\n✓ All components initialized!')

In [ ]:
# Resume from checkpoint if available
start_epoch = 0
best_iou = 0.0
loss_history = {'train': [], 'val': []}

checkpoint = ckpt_manager.load_latest()
if checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_iou = checkpoint.get('val_iou', 0.0)
    loss_history = checkpoint.get('loss_history', loss_history)
    for ep in range(start_epoch):
        scheduler.step(ep)
    print(f'✓ Resumed from epoch {start_epoch}, best IoU: {best_iou:.4f}')
else:
    print('Starting fresh training')

In [ ]:
# ═══════════════════════════════════════════════════════
# MAIN TRAINING LOOP
# ═══════════════════════════════════════════════════════

ACCUM_STEPS = CONFIG['accumulation_steps']
no_improve = 0

for epoch in range(start_epoch, CONFIG['total_epochs']):
    epoch_start = time.time()
    print(f'\n{"="*60}')
    print(f'Epoch {epoch+1}/{CONFIG["total_epochs"]} | LR: {scheduler.get_lr():.2e}')
    print(f'{"="*60}')
    
    # ── TRAIN ──
    model.train()
    train_loss = 0.0
    train_iou = 0.0
    n_train = 0
    optimizer.zero_grad()
    
    pbar = tqdm(train_loader, desc='Training', leave=False)
    for batch_idx, batch in enumerate(pbar):
        images = batch['image'].to(device, non_blocking=True)
        masks = batch['mask'].to(device, non_blocking=True)
        
        with autocast(device_type=device.type, enabled=(device.type=='cuda')):
            outputs = model(images)
            loss, _ = criterion(outputs, masks)
            loss = loss / ACCUM_STEPS
        
        scaler.scale(loss).backward()
        
        if (batch_idx + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip_norm'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        with torch.no_grad():
            main_pred = outputs[0] if isinstance(outputs, list) else outputs
            batch_iou = compute_iou(main_pred, masks)
        
        train_loss += loss.item() * ACCUM_STEPS
        train_iou += batch_iou
        n_train += 1
        
        pbar.set_postfix({
            'loss': f'{train_loss/n_train:.4f}',
            'iou': f'{train_iou/n_train:.4f}',
        })
    
    # Handle remaining gradients
    if (batch_idx + 1) % ACCUM_STEPS != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip_norm'])
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
    
    avg_train_loss = train_loss / max(n_train, 1)
    avg_train_iou = train_iou / max(n_train, 1)
    
    # ── VALIDATE ──
    model.eval()
    val_loss = 0.0
    val_iou = 0.0
    val_dice = 0.0
    n_val = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Validation', leave=False):
            images = batch['image'].to(device, non_blocking=True)
            masks = batch['mask'].to(device, non_blocking=True)
            outputs = model(images)
            loss, _ = criterion(outputs, masks)
            main_pred = outputs[0] if isinstance(outputs, list) else outputs
            val_loss += loss.item()
            val_iou += compute_iou(main_pred, masks)
            val_dice += compute_dice(main_pred, masks)
            n_val += 1
    
    avg_val_loss = val_loss / max(n_val, 1)
    avg_val_iou = val_iou / max(n_val, 1)
    avg_val_dice = val_dice / max(n_val, 1)
    
    # ── SCHEDULE ──
    scheduler.step(epoch)
    
    # ── LOG ──
    epoch_time = time.time() - epoch_start
    print(f'  Train — Loss: {avg_train_loss:.4f} | IoU: {avg_train_iou:.4f}')
    print(f'  Val   — Loss: {avg_val_loss:.4f} | IoU: {avg_val_iou:.4f} | Dice: {avg_val_dice:.4f}')
    print(f'  Time: {epoch_time:.0f}s')
    
    # ── HISTORY ──
    loss_history['train'].append({'loss': avg_train_loss, 'iou': avg_train_iou})
    loss_history['val'].append({'loss': avg_val_loss, 'iou': avg_val_iou, 'dice': avg_val_dice})
    
    # ── CHECKPOINT ──
    ckpt_manager.save(
        model, optimizer, scheduler, scaler,
        epoch, avg_val_iou, CONFIG, loss_history,
    )
    
    # ── BEST MODEL ──
    if avg_val_iou > best_iou:
        best_iou = avg_val_iou
        no_improve = 0
        print(f'  ★ New best IoU: {best_iou:.4f}')
    else:
        no_improve += 1
        print(f'  No improvement ({no_improve}/{CONFIG["early_stopping_patience"]})')
    
    # ── EARLY STOPPING ──
    if no_improve >= CONFIG['early_stopping_patience']:
        print(f'\n⚠ Early stopping at epoch {epoch+1}')
        break

print(f'\n{"="*60}')
print(f'TRAINING COMPLETE — Best Val IoU: {best_iou:.4f}')
print(f'{"="*60}')

## Section F — Training Curves & Visualization

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(loss_history['train']) + 1)

# Loss
ax1.plot(epochs_range, [m['loss'] for m in loss_history['train']], label='Train', linewidth=2)
ax1.plot(epochs_range, [m['loss'] for m in loss_history['val']], label='Val', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# IoU
ax2.plot(epochs_range, [m['iou'] for m in loss_history['train']], label='Train', linewidth=2)
ax2.plot(epochs_range, [m['iou'] for m in loss_history['val']], label='Val', linewidth=2)
ax2.axhline(y=0.85, color='green', linestyle='--', alpha=0.5, label='Target (0.85)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('IoU')
ax2.set_title('IoU Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CONFIG["checkpoint_dir"]}/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Visualize sample predictions
from data.augmentations import denormalize_image

model.eval()
batch = next(iter(val_loader))
images = batch['image'][:4].to(device)
masks = batch['mask'][:4]

with torch.no_grad():
    preds = model(images)
    if isinstance(preds, list):
        preds = preds[0]
    preds = preds.cpu()

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for i in range(4):
    img = denormalize_image(images[i].cpu())
    gt = masks[i, 0].numpy()
    pred = preds[i, 0].numpy()
    
    axes[i,0].imshow(img); axes[i,0].set_title('Input'); axes[i,0].axis('off')
    axes[i,1].imshow(gt, cmap='gray'); axes[i,1].set_title('GT'); axes[i,1].axis('off')
    axes[i,2].imshow(pred, cmap='gray'); axes[i,2].set_title(f'Pred (IoU={compute_iou(pred, gt):.3f})'); axes[i,2].axis('off')
    
    overlay = img.copy()
    overlay[(pred > 0.5) == False] = overlay[(pred > 0.5) == False] // 2
    axes[i,3].imshow(overlay); axes[i,3].set_title('Overlay'); axes[i,3].axis('off')

plt.tight_layout()
plt.show()

## Section G — Export to ONNX + Quantize

In [ ]:
# Export best model to ONNX + INT8
from export_onnx import export_to_onnx, quantize_onnx, export_torchscript

EXPORT_DIR = f'{PROJECT_DIR}/exported_models'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Load best model
best_ckpt = torch.load(f'{CONFIG["checkpoint_dir"]}/best_model.pth', map_location='cpu')
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()

# Export
onnx_path = f'{EXPORT_DIR}/model.onnx'
int8_path = f'{EXPORT_DIR}/model_int8.onnx'
ts_path = f'{EXPORT_DIR}/model.pt'

export_to_onnx(model, onnx_path, CONFIG['input_size'])
quantize_onnx(onnx_path, int8_path)
export_torchscript(model, ts_path, CONFIG['input_size'])

print(f'\n✓ Models exported to {EXPORT_DIR}')
!ls -lh {EXPORT_DIR}/

In [ ]:
# Benchmark on Colab CPU
from export_onnx import benchmark_models

model.cpu().eval()
benchmark_models(model, onnx_path, int8_path, ts_path, CONFIG['input_size'])

## Section H — Copy Results to Drive

In [ ]:
# Copy everything important to Google Drive for persistence
import shutil

# Copy checkpoints
!cp -r {CONFIG['checkpoint_dir']}/* {PROJECT_DIR}/checkpoints/ 2>/dev/null

# Copy exported models
!cp -r {EXPORT_DIR}/* {PROJECT_DIR}/exported_models/ 2>/dev/null

print(f'\n✓ All results saved to Google Drive: {PROJECT_DIR}')
!ls -lh {PROJECT_DIR}/exported_models/

## 📌 Colab Tips

**Time Limits:**
- Colab Free: ~12 hours runtime limit
- Solution: Checkpoints saved every epoch → resume next session
- Pro tip: Keep tab active, interact every 90 minutes

**Expected IoU Milestones:**
| Epochs | Expected IoU | Notes |
|--------|-------------|-------|
| 10 | 0.55-0.65 | Warmup complete, learning basic shapes |
| 30 | 0.70-0.78 | Good segmentation, rough edges |
| 50 | 0.80-0.85 | Strong results, edges improving |
| 70 | 0.85-0.90 | Best achievable from scratch |

**If training stalls:**
1. Check learning rate (should be decreasing)
2. Reduce batch size if OOM
3. Check data quality (corrupt images?)
4. Try increasing augmentation strength